In [1]:
import numpy as np
import sys
from pathlib import Path

sys.path.append('../code/')

from mlalgos import HyperOpt
from mllib import Utilities,MLUtilities

from time import time
import copy,pickle

import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib as mpl
import matplotlib.colors as pltcol
import gc

import tensorflow as tf

ut = Utilities()
ml = MLUtilities()

I0000 00:00:1783860626.335115 3593003 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783860626.336274 3593003 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783860626.407557 3593003 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783860628.061206 3593003 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True
mpl.rcParams['ytick.right'] = True
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['legend.fontsize'] = 14 # 14
mpl.rcParams['legend.labelspacing'] = 0.25
FS = 18
FS2 = 15
FS3 = 13
FSL = 22

mpl.rcParams['xtick.major.size'] = 6
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['ytick.major.size'] = 6
mpl.rcParams['ytick.minor.size'] = 3

#mpl.rcParams.keys()

# Example usage of curriculum learning  
### using shallow networks with `HyperOpt` for optimization

In [3]:
Baseline = True # set to True to establish zero noise baseline

Plot_Stem = 'curriculum/plots'
Path(Plot_Stem).mkdir(parents=True,exist_ok=True)

N_Lessons = 10 # number of lessons in training curriculum
N_Levels  = 2  # number of difficulty levels in test data

Noise_Schedule = np.logspace(-2.0,0.0,N_Lessons)
Noise_Schedule_Test = np.logspace(-2.0,0.0,N_Levels)
if Baseline:
    Noise_Schedule *= 0.0
    Noise_Schedule_Test *= 0.0

Nproc = 8

Save_Fig = True

In [4]:
Depth_Str = 'shallow'
Plot_Str = '' + Depth_Str

Family = 'seq'

## Classification example: MNIST

### Data setup with levels of increasing difficulty (noise)

In [7]:
Vortex = True
Downsample = 0.05 # 1.0 is doable on desktop: reaches ~59.6GB RAM using N = 3 x 48 jobs, scales ~ 1/N
K = 10

In [8]:
start_time = time()
(X_train_tf, Y_train_tf), (X_test_tf, Y_test_tf) = tf.keras.datasets.mnist.load_data()

print(X_train_tf.shape,Y_train_tf.shape)
print(X_test_tf.shape,Y_test_tf.shape)

if Vortex:
    print('... vortex unroll')
    X_train = ml.vortex_package(X_train_tf.T).astype(float) # (784,nsamp)
    X_test = ml.vortex_package(X_test_tf.T).astype(float) # (784,nsamp)
else:
    print('... standard flatten')
    X_train = np.zeros((X_train_tf.shape[0],X_train_tf.shape[1]**2))
    for i in range(X_train_tf.shape[0]):
        X_train[i] = X_train_tf[i].flatten()
    X_train = X_train.T
    
    X_test = np.zeros((X_test_tf.shape[0],X_test_tf.shape[1]**2))
    for i in range(X_test_tf.shape[0]):
        X_test[i] = X_test_tf[i].flatten()
    X_test = X_test.T

# don't standardize here, use 'standardize_X' instead
# X_train /= 255.0
# X_test /= 255.0
Y_train = ml.rv(Y_train_tf)
Y_test = ml.rv(Y_test_tf)

del X_train_tf,Y_train_tf
del X_test_tf,Y_test_tf
gc.collect()

eps = 1 if Y_train.min() < 1 else 0 
if K > 2:
    Y_train = ml.one_hot(Y_train+eps,K) # (K,nsamp), domain {0,1}
    Y_test = ml.one_hot(Y_test+eps,K) # (K,nsamp), domain {0,1}

print(X_train.shape,Y_train.shape)
print(X_test.shape,Y_test.shape)

rng = np.random.RandomState(1991)

if Downsample is None:
    print('... no downsampling')
    n_train = X_train.shape[1]
    n_test = X_test.shape[1]
else:
    print('... downsampling by factor {0:.3f}'.format(Downsample))
    n_train = int(Downsample*X_train.shape[1])
    n_test = int(Downsample*X_test.shape[1])  

    ind_train = rng.choice(X_train.shape[1],size=n_train,replace=False)
    ind_test = rng.choice(X_test.shape[1],size=n_test,replace=False)

    X_train = X_train[:,ind_train]
    Y_train = Y_train[:,ind_train]
    X_test = X_test[:,ind_test]
    Y_test = Y_test[:,ind_test]
    print('... retained {0:d} training and {1:d} testing samples'.format(n_train,n_test))

    del ind_train,ind_test
    gc.collect()

print('... shuffling')

ind_shuff_train = rng.choice(n_train,size=n_train,replace=False)
X_train = X_train[:,ind_shuff_train]
Y_train = Y_train[:,ind_shuff_train]

ind_shuff_test = rng.choice(n_test,size=n_test,replace=False)
X_test = X_test[:,ind_shuff_test]
Y_test = Y_test[:,ind_shuff_test]

del ind_shuff_train,ind_shuff_test
gc.collect()

for k in range(K):
    print(k,np.where(Y_train[k]==1)[0].size,np.where(Y_test[k]==1)[0].size)    


Seeds = 1983 + 2*np.arange(np.max([N_Lessons,N_Levels])) # seeds for training samples
dSeed = 5 # offset to add for test samples

print('... adding Gaussian noise')
n_train_per_lesson = n_train // N_Lessons

i_train_prev = 0
for n in range(N_Lessons):
    rng = np.random.RandomState(Seeds[n])
    
    n_train_this = 1*n_train_per_lesson
    i_train_this = n_train_per_lesson*(n+1)
    if n == (N_Lessons-1):
        n_train_this += (n_train % N_Lessons)
        i_train_this = n_train # last step should reach the end
    noise = Noise_Schedule[n]*rng.randn(n_train_this)
    X_train[:,i_train_prev:i_train_this] += noise
    i_train_prev = 1*i_train_this

i_test_prev = 0
n_test_per_level = n_test // N_Levels

for n in range(N_Levels):
    rng = np.random.RandomState(Seeds[n]+dSeed)
    n_test_this = 1*n_test_per_level
    i_test_this = n_test_per_level*(n+1)
    if n == (N_Levels-1):
        n_test_this += (n_test % N_Levels)
        i_test_this = n_test # last step should reach the end
    noise = Noise_Schedule_Test[n]*rng.randn(n_test_this)
    X_test[:,i_test_prev:i_test_this] += noise
    i_test_prev = 1*i_test_this

print('... done')
ut.time_this(start_time)

(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)
... vortex unroll
(784, 60000) (10, 60000)
(784, 10000) (10, 10000)
... downsampling by factor 0.050
... retained 3000 training and 500 testing samples
... shuffling
0 289 46
1 342 67
2 273 53
3 308 45
4 290 48
5 279 43
6 285 51
7 326 54
8 322 44
9 286 49
... adding Gaussian noise
... done
0 min 4.47 seconds



### Network setup and training
*single 2-layer network with 3 neurons and tanh activation, trained for 1000 epochs with lrate=1e-3, is enough for the standard two-moons problem*

In [9]:
# dictionary containing all setup parameters and data
setup_dict = {} 

# -- data set: features and labels
setup_dict['X'] = X_train
setup_dict['Y'] = Y_train

# -- network family
setup_dict['family'] = Family

# -- curriculum
setup_dict['curriculum'] = None # will be changed below
setup_dict['revision_frac'] = 0.1

# -- training sample 
setup_dict['train_frac'] = 0.8
setup_dict['val_frac'] = 0.2
setup_dict['loss_type'] = 'nllm'
setup_dict['neg_labels'] = False 

# -- training setup
setup_dict['standardize_X'] = True
setup_dict['max_epoch'] = 100
setup_dict['check_after'] = 30 
setup_dict['seed'] = None
setup_dict['file_stem'] = 'net' # will be changed later

#-----------------------
# total number of networks trained will be n_iter * max_config
N_Iter = 3 
Max_Config = 20
setup_dict['n_iter'] = N_Iter
setup_dict['max_config'] = Max_Config
#-----------------------

setup_dict['ensemble'] = True 
setup_dict['ensemble_size'] = 5 

setup_dict['parallel'] = True
setup_dict['nproc'] = np.min([Nproc,N_Iter*Max_Config])
setup_dict['fixed_width'] = False
setup_dict['fixed_htype'] = False

# -- sampled parameters
setup_dict['layers'] = {'min':2,'max':3}
setup_dict['widths'] = {'min':50,'max':150}
setup_dict['lglrates'] = {'min':-3.0,'max':-2.0}
setup_dict['wt_decays'] = {'min':0.0,'max':0.1}
setup_dict['thresholds'] = {'min':0.4,'max':0.6}
setup_dict['htypes'] = ['tanh','lrelu','splus','sin']
setup_dict['lrelu_slopes'] = {'min':-1e-2,'max':1e-2}
setup_dict['reg_funs'] = ['drop','none'] 
setup_dict['p_drops'] = {'min':0.0,'max':0.5}

# -- I/O
setup_dict['verbose'] = True
setup_dict['logfile'] = None

Out_Dir = 'curriculum/'
Out_Dir += 'baseline/' if Baseline else 'noisy/'
print('output dir:',Out_Dir)

output dir: curriculum/baseline/


#### Standard learning

In [ ]:
Optimize = True

File_Stem = Out_Dir + 'standard'
print('File_Stem:',File_Stem)

setup_dict_std = copy.deepcopy(setup_dict)
setup_dict_std['file_stem'] = File_Stem

start_time = time()
hopt_std = HyperOpt(setup_dict=setup_dict_std)

if Optimize:
    neo_std = hopt_std.optimize()
else:
    neo_std = hopt_std.load()

ut.time_this(start_time)

neo_std.display_summary(show_keys=['L','wt_decay','n_layer','atypes','reg_fun'])

File_Stem: curriculum/baseline/standard
--------------
Hyperparameter + architecture optimization
--------------
... neural network family: Sequential
... found data set of dimension 784 with targets of dimension 10
... found 3000 samples
... fraction 0.800 (2400 samples) will be used for training
... will search over 3 iterations of 20 configurations
... will store best 5 networks in ensemble
... will use misclassification fraction for hyperparameter comparison
... weight decays will use norm 2
... setup complete
Initiating search... 
... setting tasks
... training using 8 process(es)


#### Curriculum learning

In [ ]:
Optimize = True

File_Stem = Out_Dir + 'curriculum'
print('File_Stem:',File_Stem)

Curriculum = n_train_per_lesson*np.arange(1,N_Lessons+1)

setup_dict_curr = copy.deepcopy(setup_dict)
setup_dict_curr['file_stem'] = File_Stem
setup_dict_curr['curriculum'] = Curriculum

start_time = time()
hopt_curr = HyperOpt(setup_dict=setup_dict_curr)

if Optimize:
    neo_curr = hopt_curr.optimize()
else:
    neo_curr = hopt_curr.load()

ut.time_this(start_time)

neo_curr.display_summary(show_keys=['L','wt_decay','n_layer','atypes','reg_fun'])

### Performace

In [ ]:
Ypred_std = neo_std.predict(X_test)
Ypred_curr = neo_curr.predict(X_test)

N_ens_thresh = 4
ascs = {'easy':{'std':None,'curr':None},'hard':{'std':None,'curr':None}}

print('Assessment (ensemble average):')
print('-----------------------------')
print('Easy examples:')
print('-------------')
sl_easy = np.s_[:,:n_test_per_level]

print('... standard training')
ascs['easy']['std'] = ml.assess_multi_classification_ensemble(neo_std,X_test[sl_easy],Y_test[sl_easy],N_ens_thresh=N_ens_thresh)
for key in ['precision','recall','F1score']:
    print('... ... '+key+': {0:.5f} + {1:.5f} - {2:.5f}'.format(ascs['easy']['std'][key]['median'],
                                                                ascs['easy']['std'][key]['84pc']-ascs['easy']['std'][key]['median'],
                                                                ascs['easy']['std'][key]['median']-ascs['easy']['std'][key]['16pc']))

print('... curriculum learning')
ascs['easy']['curr'] = ml.assess_multi_classification_ensemble(neo_curr,X_test[sl_easy],Y_test[sl_easy],N_ens_thresh=N_ens_thresh)
for key in ['precision','recall','F1score']:
    print('... ... '+key+': {0:.5f} + {1:.5f} - {2:.5f}'.format(ascs['easy']['curr'][key]['median'],
                                                                ascs['easy']['curr'][key]['84pc']-ascs['easy']['curr'][key]['median'],
                                                                ascs['easy']['curr'][key]['median']-ascs['easy']['curr'][key]['16pc']))
print('-------------')

print('Hard examples:')
print('-------------')
sl_hard = np.s_[:,n_test_per_level:]

print('... standard training')
ascs['hard']['std'] = ml.assess_multi_classification_ensemble(neo_std,X_test[sl_hard],Y_test[sl_hard],N_ens_thresh=N_ens_thresh)
for key in ['precision','recall','F1score']:
    print('... ... '+key+': {0:.5f} + {1:.5f} - {2:.5f}'.format(ascs['hard']['std'][key]['median'],
                                                                ascs['hard']['std'][key]['84pc']-ascs['hard']['std'][key]['median'],
                                                                ascs['hard']['std'][key]['median']-ascs['hard']['std'][key]['16pc']))

print('... curriculum learning')
ascs['hard']['curr'] = ml.assess_multi_classification_ensemble(neo_curr,X_test[sl_hard],Y_test[sl_hard],N_ens_thresh=N_ens_thresh)
for key in ['precision','recall','F1score']:
    print('... ... '+key+': {0:.5f} + {1:.5f} - {2:.5f}'.format(ascs['hard']['curr'][key]['median'],
                                                                ascs['hard']['curr'][key]['84pc']-ascs['hard']['curr'][key]['median'],
                                                                ascs['hard']['curr'][key]['median']-ascs['hard']['curr'][key]['16pc']))
print('-------------')


#### Plot

In [ ]:
cols = ['navy','darkolivegreen','crimson','darkgoldenrod']
plt.figure(figsize=(5,3))

ticks = {'precision':[0.2,'precision'],
         'recall':[0.5,'recall'],
         'accuracy':[0.8,'accuracy']}

plt.xlim(0,1)
# ymin,ymax = (0.995,1.01) if FtypeClass=='moons' else (0.3,1.1)
# plt.ylim(ymin,ymax)
plt.xticks([ticks[key][0] for key in ticks.keys()],
           [ticks[key][1] for key in ticks.keys()],rotation=30)

for key in ticks.keys():
    cc = 0
    for diff in ['easy','hard']:
        for learn in ['std','curr']:
            asc = ascs[diff][learn]
            Label = (learn + ',' + diff) if key=='precision' else None
            plt.errorbar([ticks[key][0]],[asc[key]['median']],
                         yerr=[[asc[key]['median']-asc[key]['16pc']],[asc[key]['84pc']-asc[key]['median']]],
                         c=cols[cc],
                         capsize=5,marker='o',ls='none',markersize=5,label=Label)
            cc += 1
    
# xtext = 0.05
# ytext1 = 1.008 if FtypeClass=='moons' else 1.0
# ytext2 = 1.006 if FtypeClass=='moons' else 0.9
# ytext3 = 1.0045 if FtypeClass=='moons' else 0.825
# plt.text(xtext,ytext1,'MNIST ({0:d})'.format(K),fontsize=FS3)
# plt.text(xtext,ytext2,Depth_Str_Root+' nets',fontsize=FS3)
# plt.text(xtext,ytext3,Std_Text,fontsize=FS3)
# ytext4 = 0.8 
# ytext5 = 0.7
# plt.text(0.55,ytext5,'$\\langle \\, N_{{\\rm wts}} \\, \\rangle = {0:.1f}$'.format(ensc_avg_Nwts),fontsize=FS3,c=cols[1])

plt.legend(loc='upper right')
plt.axhline(1.0,c='k',ls=':',lw=1)
if Save_Fig:
    outfile = Plot_Stem + '/performance'
    outfile += '_baseline' if Baseline else '_noisy'
    outfile += '.pdf'
    print('Writing to file: '+outfile)
    plt.savefig(outfile,bbox_inches='tight')
else:
    plt.show()